In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
    roc_curve,
    precision_recall_curve
)

from xgboost import XGBClassifier


# ============================================================
# CONFIGURAÇÕES
# ============================================================

dataset = "df_treino_quadrilatero_v4"

caminho_dataset = Path("../datasets") / f"{dataset}.csv"
pasta_resultados = Path(f"{dataset}-resultados")
pasta_resultados.mkdir(exist_ok=True)

arquivo_saida = pasta_resultados / f"{dataset}-resultados.txt"

thresholds = [0.50, 0.45, 0.40, 0.35, 0.30]

# Ajuste esta lista caso existam outras variáveis de topografia
variaveis_topografia = [
    "nome_entrada",
    "desenv_aproximado",
    "desnivel_propeccao"
]


# ============================================================
# LEITURA E PREPARAÇÃO DOS DADOS
# ============================================================

df_treino = pd.read_csv(caminho_dataset)

df_treino = df_treino.drop(
    columns=["cavidade"],
    errors="ignore"
)

# Remove target nulo ou vazio, por segurança
df_treino["target"] = df_treino["target"].replace(r"^\s*$", np.nan, regex=True)
df_treino = df_treino.dropna(subset=["target"]).copy()

df_treino["target"] = pd.to_numeric(
    df_treino["target"],
    errors="raise"
).astype(int)

y = df_treino["target"].copy()
X = df_treino.drop(columns=["target"]).copy()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)
X_train.to_excel(f"dataset_treino_{dataset}.xlsx")
X_test.to_excel(f"dataset_teste_{dataset}.xlsx")
# ============================================================
# MODELOS
# ============================================================

modelos = {
    "XGBoost": XGBClassifier(
        n_estimators=1000,
        max_depth=3,
        learning_rate=0.03,
        min_child_weight=5,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        eval_metric="logloss"
    ),

    "Decision Tree": DecisionTreeClassifier(
        max_depth=20,
        random_state=42,
        class_weight="balanced"
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=1400,
        random_state=42,
        class_weight="balanced",
        n_jobs=-1
    )
}


# ============================================================
# FUNÇÕES AUXILIARES
# ============================================================

def nome_seguro(texto):
    """
    Transforma o nome do modelo em um nome apropriado para arquivo.
    """

    return (
        texto.lower()
        .replace(" ", "_")
        .replace("-", "_")
    )


def interpretar_matriz(tn, fp, fn, tp):
    """
    Gera uma interpretação textual simples da matriz de confusão.
    """

    total = tn + fp + fn + tp

    positivos_reais = tp + fn
    negativos_reais = tn + fp
    positivos_previstos = tp + fp

    recall = tp / positivos_reais if positivos_reais > 0 else 0
    especificidade = tn / negativos_reais if negativos_reais > 0 else 0
    precision = tp / positivos_previstos if positivos_previstos > 0 else 0
    acuracia = (tp + tn) / total if total > 0 else 0

    return {
        "acuracia": acuracia,
        "precision": precision,
        "recall": recall,
        "especificidade": especificidade
    }


def salvar_matriz_confusao(
    matriz,
    nome_modelo,
    threshold,
    caminho_saida
):
    """
    Salva a matriz de confusão bruta em PNG.
    """

    fig, ax = plt.subplots(figsize=(6, 5))

    imagem = ax.imshow(
        matriz,
        cmap="Blues"
    )

    ax.set_title(
        f"{nome_modelo}\nMatriz de confusão - threshold {threshold:.2f}"
    )

    ax.set_xlabel("Classe prevista")
    ax.set_ylabel("Classe real")

    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])

    ax.set_xticklabels(["0", "1"])
    ax.set_yticklabels(["0", "1"])

    for i in range(2):
        for j in range(2):
            ax.text(
                j,
                i,
                str(matriz[i, j]),
                ha="center",
                va="center",
                fontsize=14,
                fontweight="bold"
            )

    fig.colorbar(imagem, ax=ax)

    plt.tight_layout()
    plt.savefig(
        caminho_saida,
        dpi=150,
        bbox_inches="tight"
    )
    plt.close()


def obter_importancia_variaveis(
    modelo,
    nome_modelo,
    X_test,
    y_test
):
    """
    Calcula a importância das variáveis.

    Utiliza permutation importance para permitir uma comparação
    padronizada entre os três modelos.
    """

    resultado = permutation_importance(
        estimator=modelo,
        X=X_test,
        y=y_test,
        scoring="average_precision",
        n_repeats=20,
        random_state=42,
        n_jobs=-1
    )

    importancia = pd.DataFrame({
        "modelo": nome_modelo,
        "variavel": X_test.columns,
        "importancia_media": resultado.importances_mean,
        "desvio_padrao": resultado.importances_std
    })

    importancia["topografia"] = importancia["variavel"].isin(
        variaveis_topografia
    )

    importancia = importancia.sort_values(
        "importancia_media",
        ascending=False
    ).reset_index(drop=True)

    return importancia


def salvar_grafico_importancia(
    importancia,
    nome_modelo,
    caminho_saida,
    quantidade=15
):
    """
    Salva o gráfico com as variáveis mais importantes.
    """

    dados = importancia.head(quantidade).sort_values(
        "importancia_media",
        ascending=True
    )

    cores = np.where(
        dados["topografia"],
        "darkorange",
        "steelblue"
    )

    plt.figure(figsize=(10, 7))

    plt.barh(
        dados["variavel"],
        dados["importancia_media"],
        xerr=dados["desvio_padrao"],
        color=cores,
        alpha=0.85
    )

    plt.title(
        f"{nome_modelo}\nImportância por permutação"
    )

    plt.xlabel(
        "Queda média no Average Precision após embaralhar a variável"
    )

    plt.ylabel("Variável")

    plt.tight_layout()

    plt.savefig(
        caminho_saida,
        dpi=150,
        bbox_inches="tight"
    )

    plt.close()


# ============================================================
# EXECUÇÃO E AVALIAÇÃO
# ============================================================

resultados_thresholds = []
resultados_modelos = []
importancias_completas = []

curvas_roc = {}
curvas_pr = {}

with open(arquivo_saida, "w", encoding="utf-8") as f:

    def log(text=""):
        print(text)
        f.write(str(text) + "\n")


    # --------------------------------------------------------
    # INFORMAÇÕES DO DATASET
    # --------------------------------------------------------

    log("=" * 80)
    log("INFORMAÇÕES DO DATASET")
    log("=" * 80)

    log(f"Dataset: {dataset}")
    log(f"Total de registros antes do split: {len(df_treino)}")
    log(f"Total de variáveis preditoras: {X.shape[1]}")

    log(f"Tamanho de X completo: {X.shape}")
    log(f"Tamanho de y completo: {y.shape}")

    log(f"Registros de treinamento: {len(X_train)}")
    log(f"Registros de teste: {len(X_test)}")

    log(
        f"Percentual efetivo de treinamento: "
        f"{len(X_train) / len(X):.2%}"
    )

    log(
        f"Percentual efetivo de teste: "
        f"{len(X_test) / len(X):.2%}"
    )

    log("\nDistribuição completa do target:")
    log(y.value_counts().sort_index().to_string())

    log("\nDistribuição percentual completa:")
    log(
        y.value_counts(
            normalize=True
        ).sort_index().mul(100).round(2).to_string()
    )

    log("\nDistribuição do target no treino:")
    log(y_train.value_counts().sort_index().to_string())

    log("\nDistribuição do target no teste:")
    log(y_test.value_counts().sort_index().to_string())

    log()


    # --------------------------------------------------------
    # TREINAMENTO DOS MODELOS
    # --------------------------------------------------------

    for nome_modelo, modelo in modelos.items():

        log("=" * 80)
        log(nome_modelo.upper())
        log("=" * 80)

        modelo.fit(X_train, y_train)

        y_prob = modelo.predict_proba(X_test)[:, 1]

        roc_auc = roc_auc_score(
            y_test,
            y_prob
        )

        average_precision = average_precision_score(
            y_test,
            y_prob
        )

        fpr, tpr, thresholds_roc = roc_curve(
            y_test,
            y_prob
        )

        precision_curve, recall_curve, thresholds_pr = (
            precision_recall_curve(
                y_test,
                y_prob
            )
        )

        curvas_roc[nome_modelo] = {
            "fpr": fpr,
            "tpr": tpr,
            "roc_auc": roc_auc
        }

        curvas_pr[nome_modelo] = {
            "precision": precision_curve,
            "recall": recall_curve,
            "average_precision": average_precision
        }

        log(f"ROC-AUC: {roc_auc:.4f}")
        log(f"Average Precision: {average_precision:.4f}")

        resultados_modelos.append({
            "dataset": dataset,
            "modelo": nome_modelo,
            "roc_auc": roc_auc,
            "average_precision": average_precision,
            "registros_total": len(df_treino),
            "registros_treino": len(X_train),
            "registros_teste": len(X_test),
            "quantidade_variaveis": X.shape[1]
        })


        # ----------------------------------------------------
        # RESULTADOS POR THRESHOLD
        # ----------------------------------------------------

        for threshold in thresholds:

            y_pred = (
                y_prob >= threshold
            ).astype(int)

            matriz = confusion_matrix(
                y_test,
                y_pred,
                labels=[0, 1]
            )

            tn, fp, fn, tp = matriz.ravel()

            metricas = interpretar_matriz(
                tn,
                fp,
                fn,
                tp
            )

            log()
            log(
                f"{nome_modelo} - Dataset {dataset}"
            )

            log(f"Threshold: {threshold:.2f}")

            log("\nMatriz de confusão bruta:")
            log(matriz)

            log(f"Verdadeiros negativos (TN): {tn}")
            log(f"Falsos positivos (FP): {fp}")
            log(f"Falsos negativos (FN): {fn}")
            log(f"Verdadeiros positivos (TP): {tp}")

            log(
                f"Acurácia calculada: "
                f"{metricas['acuracia']:.4f}"
            )

            log(
                f"Precisão da classe positiva: "
                f"{metricas['precision']:.4f}"
            )

            log(
                f"Recall da classe positiva: "
                f"{metricas['recall']:.4f}"
            )

            log(
                f"Especificidade da classe negativa: "
                f"{metricas['especificidade']:.4f}"
            )

            log("\nClassification report:")

            log(
                classification_report(
                    y_test,
                    y_pred,
                    digits=4,
                    zero_division=0
                )
            )

            resultados_thresholds.append({
                "dataset": dataset,
                "modelo": nome_modelo,
                "threshold": threshold,
                "tn": tn,
                "fp": fp,
                "fn": fn,
                "tp": tp,
                "acuracia": metricas["acuracia"],
                "precision_classe_1": metricas["precision"],
                "recall_classe_1": metricas["recall"],
                "especificidade_classe_0": metricas["especificidade"],
                "roc_auc": roc_auc,
                "average_precision": average_precision
            })

            nome_arquivo_matriz = (
                f"{nome_seguro(nome_modelo)}"
                f"_matriz_threshold_{threshold:.2f}.png"
            )

            salvar_matriz_confusao(
                matriz=matriz,
                nome_modelo=nome_modelo,
                threshold=threshold,
                caminho_saida=(
                    pasta_resultados
                    / nome_arquivo_matriz
                )
            )

            log("-" * 80)


        # ----------------------------------------------------
        # IMPORTÂNCIA DAS VARIÁVEIS
        # ----------------------------------------------------

        importancia = obter_importancia_variaveis(
            modelo=modelo,
            nome_modelo=nome_modelo,
            X_test=X_test,
            y_test=y_test
        )

        importancias_completas.append(importancia)

        log("\nIMPORTÂNCIA DAS VARIÁVEIS")
        log(
            importancia.head(15).to_string(
                index=False
            )
        )

        importancia_topografia = importancia[
            importancia["topografia"]
        ]

        log("\nIMPORTÂNCIA DAS VARIÁVEIS DE TOPOGRAFIA")

        if importancia_topografia.empty:
            log(
                "Nenhuma variável configurada como topografia "
                "foi encontrada no dataset."
            )
        else:
            log(
                importancia_topografia.to_string(
                    index=False
                )
            )

        salvar_grafico_importancia(
            importancia=importancia,
            nome_modelo=nome_modelo,
            caminho_saida=(
                pasta_resultados
                / f"{nome_seguro(nome_modelo)}_importancia.png"
            )
        )

        log()


    # --------------------------------------------------------
    # COMPARAÇÃO FINAL
    # --------------------------------------------------------

    df_resultados_modelos = pd.DataFrame(
        resultados_modelos
    ).sort_values(
        "average_precision",
        ascending=False
    )

    log("=" * 80)
    log("COMPARAÇÃO FINAL DOS MODELOS")
    log("=" * 80)

    log(
        df_resultados_modelos.to_string(
            index=False
        )
    )

    melhor_roc = df_resultados_modelos.loc[
        df_resultados_modelos["roc_auc"].idxmax()
    ]

    melhor_pr = df_resultados_modelos.loc[
        df_resultados_modelos["average_precision"].idxmax()
    ]

    log()
    log(
        f"Melhor ROC-AUC: {melhor_roc['modelo']} "
        f"({melhor_roc['roc_auc']:.4f})"
    )

    log(
        f"Melhor Average Precision: {melhor_pr['modelo']} "
        f"({melhor_pr['average_precision']:.4f})"
    )


# ============================================================
# SALVAR RESULTADOS TABULARES
# ============================================================

df_resultados_thresholds = pd.DataFrame(
    resultados_thresholds
)

df_resultados_modelos = pd.DataFrame(
    resultados_modelos
)

df_importancias = pd.concat(
    importancias_completas,
    ignore_index=True
)

df_resultados_thresholds.to_csv(
    pasta_resultados / "resultados_por_threshold.csv",
    index=False,
    encoding="utf-8-sig"
)

df_resultados_modelos.to_csv(
    pasta_resultados / "comparacao_modelos.csv",
    index=False,
    encoding="utf-8-sig"
)

df_importancias.to_csv(
    pasta_resultados / "importancia_variaveis.csv",
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# CURVA ROC COMPARATIVA
# ============================================================

plt.figure(figsize=(9, 7))

for nome_modelo, dados in curvas_roc.items():

    plt.plot(
        dados["fpr"],
        dados["tpr"],
        linewidth=2,
        label=(
            f"{nome_modelo} "
            f"(AUC = {dados['roc_auc']:.3f})"
        )
    )

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    color="gray",
    label="Classificador aleatório"
)

plt.xlabel("Taxa de falsos positivos")
plt.ylabel("Taxa de verdadeiros positivos")
plt.title("Curva ROC comparativa")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

plt.savefig(
    pasta_resultados / "curva_roc_comparativa.png",
    dpi=150,
    bbox_inches="tight"
)

plt.close()


# ============================================================
# CURVA PRECISION-RECALL COMPARATIVA
# ============================================================

plt.figure(figsize=(9, 7))

for nome_modelo, dados in curvas_pr.items():

    plt.plot(
        dados["recall"],
        dados["precision"],
        linewidth=2,
        label=(
            f"{nome_modelo} "
            f"(AP = {dados['average_precision']:.3f})"
        )
    )

proporcao_positivos = y_test.mean()

plt.axhline(
    y=proporcao_positivos,
    linestyle="--",
    color="gray",
    label=(
        f"Baseline da classe positiva "
        f"({proporcao_positivos:.3f})"
    )
)

plt.xlabel("Recall")
plt.ylabel("Precisão")
plt.title("Curva Precision-Recall comparativa")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

plt.savefig(
    pasta_resultados / "curva_precision_recall_comparativa.png",
    dpi=150,
    bbox_inches="tight"
)

plt.close()


print()
print("Processamento concluído.")
print(f"Resultados salvos em: {pasta_resultados.resolve()}")

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
    roc_curve,
    precision_recall_curve
)

from xgboost import XGBClassifier


# ============================================================
# CONFIGURAÇÕES
# ============================================================

dataset = "df_treino_carajas_v3"

caminho_dataset = Path("../datasets") / f"{dataset}.csv"
pasta_resultados = Path(f"{dataset}-resultados")
pasta_resultados.mkdir(exist_ok=True)

arquivo_saida = pasta_resultados / f"{dataset}-resultados.txt"

thresholds = [0.50, 0.45, 0.40, 0.35, 0.30]

# Ajuste esta lista caso existam outras variáveis de topografia
variaveis_topografia = [
    "nome_entrada",
    "desenv_aproximado",
    "desnivel_propeccao"
]


# ============================================================
# LEITURA E PREPARAÇÃO DOS DADOS
# ============================================================

df_treino = pd.read_csv(caminho_dataset)

df_treino = df_treino.drop(
    columns=["cavidade"],
    errors="ignore"
)

# Remove target nulo ou vazio, por segurança
df_treino["target"] = df_treino["target"].replace(r"^\s*$", np.nan, regex=True)
df_treino = df_treino.dropna(subset=["target"]).copy()

df_treino["target"] = pd.to_numeric(
    df_treino["target"],
    errors="raise"
).astype(int)

y = df_treino["target"].copy()
X = df_treino.drop(columns=["target"]).copy()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)
X_train.to_excel(f"dataset_treino_{dataset}.xlsx")
X_test.to_excel(f"dataset_teste_{dataset}.xlsx")

In [ ]:
import pandas as pd

from xgboost import XGBClassifier

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    accuracy_score
)

# ==========================================================
# CONFIGURAÇÕES
# ==========================================================

SEEDS = range(100)

THRESHOLDS = [
    0.50,
    0.45,
    0.40,
    0.35,
    0.30
]
dataset = "df_treino_quadrilatero_v4"
# ==========================================================
# EXECUÇÃO
# ==========================================================

resultados = []

for seed in SEEDS:

    print(f"Seed {seed}")

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        stratify=y,
        random_state=seed
    )

    modelos = {
        "XGBoost": XGBClassifier(
            n_estimators=1000,
            max_depth=3,
            learning_rate=0.03,
            min_child_weight=5,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42
        ),

        "Decision Tree": DecisionTreeClassifier(
            max_depth=20,
            random_state=42,
            class_weight="balanced"
        ),

        "Random Forest": RandomForestClassifier(
            n_estimators=1400,
            random_state=42,
            class_weight="balanced",
            n_jobs=-1
        )
    }

    for nome_modelo, modelo in modelos.items():

        modelo.fit(X_train, y_train)

        y_prob = modelo.predict_proba(X_test)[:, 1]

        for threshold in THRESHOLDS:

            y_pred = (y_prob >= threshold).astype(int)

            resultados.append({
                "modelo": nome_modelo,
                "seed": seed,
                "threshold": threshold,

                "precision": precision_score(
                    y_test,
                    y_pred,
                    zero_division=0
                ),

                "recall": recall_score(
                    y_test,
                    y_pred,
                    zero_division=0
                ),

                "f1": f1_score(
                    y_test,
                    y_pred,
                    zero_division=0
                ),

                "accuracy": accuracy_score(
                    y_test,
                    y_pred
                )
            })

# ==========================================================
# DETALHADO
# ==========================================================

df_detalhado = pd.DataFrame(resultados)

df_detalhado.to_excel(
    f"{dataset}_estabilidade_detalhado.xlsx",
    index=False
)

# ==========================================================
# RESUMO
# ==========================================================

linhas_resumo = []

for (modelo, threshold), grupo in df_detalhado.groupby(
    ["modelo", "threshold"]
):

    for metrica in [
        "precision",
        "recall",
        "f1",
        "accuracy"
    ]:

        media = grupo[metrica].mean()
        desvio = grupo[metrica].std()
        minimo = grupo[metrica].min()
        maximo = grupo[metrica].max()
        amplitude = maximo - minimo

        cv = 0

        if media > 0:
            cv = (desvio / media) * 100

        linhas_resumo.append({
            "modelo": modelo,
            "threshold": threshold,
            "metrica": metrica,

            "media": round(media, 4),
            "desvio_padrao": round(desvio, 4),
            "cv_percent": round(cv, 2),

            "minimo": round(minimo, 4),
            "maximo": round(maximo, 4),
            "amplitude": round(amplitude, 4)
        })

df_resumo = pd.DataFrame(linhas_resumo)

df_resumo.to_excel(
    f"{dataset}_estabilidade_resumo.xlsx",
    index=False
)

print("Arquivos gerados com sucesso.")